# Deep Learning Fraud Detection System
## Sequential Financial Transactions

**Dataset:** Credit Card Fraud Detection (Kaggle)

| Task | Topic |
|------|-------|
| 1 | Business Understanding |
| 2 | Exploratory Analysis |
| 3 | Sequence Generation |
| 4 | Model Comparison |
| 5 | Positional Encoding |
| 6 | Attention Investigation |

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.eda import compute_class_stats, plot_amount_distribution, plot_correlation_heatmap, plot_class_balance
from src.preprocessing import prepare_datasets
from src.positional_encoding import visualize_pe_heatmap
from src.attention_utils import get_transaction_importance, rank_influential_transactions, visualize_attention_bar

DATA_PATH = ROOT / 'creditcard.csv'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Task 1: Business Understanding

### Why is fraud detection difficult?
1. **Extreme class imbalance** — Fraud is < 0.2% of transactions
2. **Evolving patterns** — Fraudsters adapt continuously
3. **Sequential context** — Single txn may look normal; sequence may not
4. **High error cost** — False negatives lose money; false positives lose customers

### Why is accuracy alone misleading?
A model predicting 'not fraud' always gets ~99.8% accuracy but catches **zero fraud**.
Use **Precision, Recall, F1, ROC-AUC** instead.

## Task 2: Exploratory Analysis

In [ ]:
df = pd.read_csv(DATA_PATH)
stats = compute_class_stats(df)

print(f'Total Transactions : {stats["total_transactions"]:,}')
print(f'Fraud %            : {stats["fraud_pct"]:.4f}%')
print(f'Legitimate %       : {stats["legit_pct"]:.4f}%')
print(f'Imbalance Ratio    : {stats["imbalance_ratio"]:.1f}:1')

In [ ]:
plot_class_balance(stats); plt.show()
plot_amount_distribution(df); plt.show()
plot_correlation_heatmap(df); plt.show()

## Task 3: Sequence Generation

Txn1 → Txn2 → Txn3 → Txn4 → **Predict Txn5 fraud?**

In [ ]:
data = prepare_datasets(DATA_PATH, max_samples=30000)
print(f'Train: {data["X_train"].shape} | Fraud rate: {data["y_train"].mean():.4f}')

## Task 4: Model Comparison

Run `uv run python -m src.train` to train all models.

In [ ]:
import json
comparison_path = OUTPUT_DIR / 'model_comparison.json'
if comparison_path.exists():
    with open(comparison_path) as f:
        comparison = json.load(f)
    pd.DataFrame(comparison).T[['accuracy','precision','recall','f1','roc_auc']].round(4)
else:
    print('Run: uv run python -m src.train')

## Task 5: Positional Encoding

Order matters: card testing (small→large), velocity attacks, dormant→active patterns.

In [ ]:
visualize_pe_heatmap(seq_len=4, d_model=32); plt.show()

## Task 6: Attention Investigation

In [ ]:
import tensorflow as tf
from src.models import TransactionPositionalEncoding

extractor_path = ROOT / 'models' / 'attention_extractor.keras'
if extractor_path.exists():
    extractor = tf.keras.models.load_model(extractor_path, custom_objects={'TransactionPositionalEncoding': TransactionPositionalEncoding})
    sample = data['X_test'][:1]
    prob, attn = extractor.predict(sample, verbose=0)
    importance = get_transaction_importance(attn)[0]
    labels = [f'Txn{i+1}' for i in range(len(importance))]
    ranked = rank_influential_transactions(importance, labels)
    print(f'Fraud prob: {prob[0][0]:.4f} | Top: {ranked[0]}')
    visualize_attention_bar(importance, labels); plt.show()
else:
    print('Train first: uv run python -m src.train')

## Task 7: Dashboard

```bash
uv run streamlit run app/streamlit_app.py
```